# 🔍 Reconhecimento de Objetos: Detecção e Localização

**Reconhecimento de Objetos** é uma das tarefas mais fundamentais e desafiadoras na Visão Computacional. Ele vai além da simples classificação de imagens (que apenas diz o que está na imagem) para **detectar e localizar** um ou mais objetos dentro de uma imagem ou vídeo.

## O Que Envolve?

Para cada objeto detectado, o sistema de reconhecimento de objetos normalmente fornece:
1.  **Classe do Objeto:** Qual é o objeto (ex: "pessoa", "carro", "cachorro").
2.  **Caixa Delimitadora (Bounding Box):** As coordenadas retangulares que envolvem o objeto na imagem.
3.  **Pontuação de Confiança:** A probabilidade de que a detecção esteja correta.

## Como Funciona (Brevemente)?

Modelos modernos de reconhecimento de objetos, como YOLO (You Only Look Once), SSD (Single Shot MultiBox Detector) e Faster R-CNN, utilizam arquiteturas de Deep Learning (principalmente CNNs) para realizar a detecção. Eles são treinados em grandes conjuntos de dados (como COCO, ImageNet) que contêm milhões de imagens com objetos anotados manualmente.

Neste exemplo, usaremos o **YOLOv8**, uma versão recente e performática da família YOLO, que é conhecida pela sua velocidade e precisão.

---


In [ ]:
import torch
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

# import urllib.request
# import os
# import requests # Mesmo não usado para o download atual, é útil para downloads robustos.
# import numpy as np 

In [ ]:
# --- Função de Detecção de Objetos com YOLOv8 ---

def detect_objects_yolo(
    image_path,
    model_path="yolov8n.pt", # Modelo YOLOv8 a ser usado (ex: yolov8n.pt, yolov8s.pt, etc.)
    conf=0.25,               # Limiar de confiança para detecções (entre 0 e 1). Padrão alterado para um valor mais comum.
    imgsz=640,               # Tamanho da imagem para inferência (ex: 640, 1280). Padrão alterado para um valor mais comum.
    print_device=False,      # Se True, imprime o dispositivo usado (cuda/cpu).
    print_summary=True,      # Se True, imprime um resumo consolidado dos objetos detectados. (Padrão alterado para True)
    print_individual_detections=False # Se True, imprime cada objeto detectado individualmente (muitos prints).
):
    """
    Detecta objetos em uma imagem usando YOLOv8.
    
    Parâmetros:
        image_path (str): Caminho completo para a imagem local.
        model_path (str): Nome ou caminho para o arquivo do modelo YOLO (ex: "yolov8n.pt").
                          Se o arquivo não existir localmente, a biblioteca tentará baixá-lo.
        conf (float): Limiar de confiança (0-1). Detecções com score abaixo deste valor serão ignoradas.
                      Valores menores = mais detecções (incluindo falsos positivos).
                      Valores maiores = menos detecções (apenas as mais confiáveis).
        imgsz (int): Dimensão da imagem para redimensionamento antes da inferência. 
                     (ex: 640 para 640x640, 1280 para 1280x1280).
                     Valores maiores ajudam a detectar objetos pequenos, mas são mais lentos.
        print_device (bool): Se True, exibe qual dispositivo (GPU/CPU) está sendo usado.
        print_summary (bool): Se True, exibe um resumo consolidado das classes e quantidades detectadas.
        print_individual_detections (bool): Se True, exibe uma lista detalhada de cada objeto detectado.
    
    Retorna:
        ultralytics.engine.results.Results object: Objeto contendo todos os resultados da detecção.
    """
    
    # 1. Seleciona o dispositivo (GPU ou CPU)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if print_device:
        print(f"Dispositivo usado: {device}")

    # 2. Carrega o modelo YOLO
    # A Ultralytics baixará o modelo automaticamente se não estiver presente.
    model = YOLO(model_path)
    model.to(device) # Move o modelo para o dispositivo selecionado (CPU ou GPU)

    # 3. Carrega a imagem
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"❌ Erro: Não foi possível carregar a imagem em: {image_path}. Verifique o caminho.")
    
    # 4. Executa a detecção
    # verbose=False é crucial para suprimir os logs internos da biblioteca Ultralytics
    results = model(img, imgsz=imgsz, conf=conf, device=device, verbose=False) 
    
    # Pega o resultado da primeira (e única) imagem no batch
    res = results[0] 

    # 5. Mostra a imagem anotada com as detecções
    annotated_img = res.plot() # Gera uma imagem com as caixas delimitadoras e labels desenhadas
    
    plt.figure(figsize=(12, 8)) # Cria uma figura Matplotlib para exibir a imagem
    # Converte de BGR (formato padrão do OpenCV) para RGB (formato esperado pelo Matplotlib)
    plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)) 
    plt.axis('off') # Remove os eixos e ticks para uma visualização mais limpa
    plt.title(f"Detecção de Objetos YOLOv8 (Conf: >{conf:.2f}, ImgSize: {imgsz})") # Título da janela
    plt.show() # Exibe a janela com a imagem

    # 6. Controle dos Prints de Detecção no Console
    if print_summary or print_individual_detections:
        detected_objects_count = {}
        detected_objects_details = []

        if res.boxes is not None and len(res.boxes) > 0: # Garante que há boxes para processar
            for i, cls_id in enumerate(res.boxes.cls.tolist()):
                class_name = res.names[int(cls_id)]
                confidence = float(res.boxes.conf[i])
                
                detected_objects_count[class_name] = detected_objects_count.get(class_name, 0) + 1
                detected_objects_details.append(f"  - {class_name} (confiança: {confidence:.2f})")

        if print_summary:
            summary_str_parts = []
            for class_name, count in detected_objects_count.items():
                summary_str_parts.append(f"{count} {class_name}{'s' if count > 1 else ''}")
            
            if summary_str_parts:
                print(f"Objetos detectados: {', '.join(summary_str_parts)}")
            else:
                print("Nenhum objeto detectado com a confiança especificada.")

        if print_individual_detections:
            print("\nDetalhes de cada objeto detectado (classe e confiança):")
            if detected_objects_details:
                for detail in detected_objects_details:
                    print(detail)
            else:
                print("  Nenhum detalhe individual para exibir.")

    return res # Retorna o objeto Results para uso posterior, se necessário

In [ ]:
if __name__ == "__main__":
    # <<<<<<<<<<<<<<< ATENÇÃO: COLOQUE O CAMINHO CORRETO DA SUA IMAGEM AQUI >>>>>>>>>>>>>>>
    my_image_path = r"imagem2.jpg"

    # --- Exemplo 1: Configuração para um console muito limpo (apenas a imagem) ---
    print("--- Exemplo 1: Console limpo, com a imagem e detecções ---")
    # A chamada padrão agora tem print_summary=True e conf=0.25, imgsz=640
    # Se quiser o console *completamente* limpo, defina print_summary=False.
    results_clean = detect_objects_yolo(
        image_path=my_image_path,
        conf=0.25,      # Usará o padrão (0.25)
        imgsz=1920,   
        # print_summary=True,
        print_device=False,
        print_individual_detections=False
    )
    print("\n") 

    # --- Exemplo 2: Ajustando conf para ser mais rigoroso e imgsz maior ---
    print("--- Exemplo 2: conf=0.7 (mais rigoroso), imgsz=1280 (melhor para objetos pequenos) ---")
    results_high_conf_large_img = detect_objects_yolo(
        image_path=my_image_path,
        conf=0.7,    # Apenas detecções com 70% ou mais de confiança
        imgsz=1920,  # Redimensiona para 1280x1280 antes da inferência
        print_device=True, # Exibe o dispositivo
        # print_summary=False 
    )
    print("\n")

    # --- Exemplo 3: Ajustando conf para ser mais permissivo e imgsz padrão ---
    print("--- Exemplo 3: conf=0.05 (muito permissivo), imgsz=640 (padrão) ---")
    results_low_conf_default_img = detect_objects_yolo(
        image_path=my_image_path,
        conf=0.05, # Detecções com 5% ou mais de confiança (muito permissivo)
        imgsz=1920, # Usa o tamanho padrão
        print_device=True,
        # print_summary=False,
        print_individual_detections=True # Exibe cada detecção individualmente
    )
    print("\n")

    # Os objetos 'results_clean', 'results_high_conf_large_img', etc.,
    # contêm todas as informações das detecções e podem ser usados para processamento posterior.
    # Por exemplo, para acessar as caixas delimitadoras do último exemplo:
    # if results_low_conf_default_img.boxes is not None:
    #     print(f"Número total de caixas detectadas no Exemplo 3: {len(results_low_conf_default_img.boxes)}")

In [1]:
import torch
from ultralytics import YOLO
import cv2
import time

# Configurações
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo usado:", device)

# Carrega o modelo YOLOv8
model = YOLO("yolov8n.pt")
model.to(device)

# Inicializa a câmera
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise IOError("Não foi possível abrir a câmera")

# Tempo total de captura (em segundos)
total_time = 240  # 4 minutos
# Intervalo entre processamentos (em segundos)
interval = 5

start_time = time.time()

while True:
    current_time = time.time()
    elapsed_time = current_time - start_time
    if elapsed_time > total_time:
        break

    ret, frame = cap.read()
    if not ret:
        break

    # Processa a detecção
    results = model(frame, device=device)
    annotated_frame = results[0].plot()

    # Mostra a imagem
    cv2.imshow("YOLOv8 - Detecção (atualiza a cada 5s)", annotated_frame)

    # Espera o intervalo configurado (5 segundos) antes da próxima captura
    if cv2.waitKey(interval * 1000) & 0xFF == ord('q'):
        break

# Libera a câmera e fecha janelas
cap.release()
cv2.destroyAllWindows()


Dispositivo usado: cuda

0: 480x640 (no detections), 34.0ms
Speed: 12.9ms preprocess, 34.0ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 72.0ms
Speed: 4.3ms preprocess, 72.0ms inference, 4.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 65.0ms
Speed: 3.6ms preprocess, 65.0ms inference, 13.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 11.8ms
Speed: 2.0ms preprocess, 11.8ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 108.1ms
Speed: 3.6ms preprocess, 108.1ms inference, 3.4ms postprocess per image at shape (1, 3, 480, 640)


KeyboardInterrupt: 